# 🎯 Notebook 05: 端到端基线管道

## 学习目标
1. 理解"端到端 (E2E) 管道"的概念
2. 掌握持续法预测 (Persistence Forecast) 的原理
3. 理解 P&L (盈亏) 计算在电力交易中的意义
4. 看到数据→预测→交易→盈亏 的完整链路

## 为什么这个是 Notebook 05 而不是最后？

按照先精深再整合的直觉，端到端应该放在最后。
但研究（PITFALLS.md）揭示了最常见的陷阱:

> **Pitfall #5: "完美模型，没有集成"陷阱**
> 学习者在孤岛上花数周优化预测精度，
> 但从没连接到仿真系统中跑一次。

**解决方案: 先跑通，再精深。**
用最简单的模型（持续法）证明整个管道是通的，
然后再用 XGBoost 替换预测环节。

这叫做 **Walking Skeleton (行走骨架)** ——
软件工程中的最佳实践：先把骨架搭好走起来，再添肉。

In [ ]:
# ── 导入依赖 ──────────────────────────────────
import sys
sys.path.insert(0, '..')

from pipeline.data_loader import create_loader
from pipeline.cleaner import clean_data
from pipeline.forecaster import persistence_forecast, calculate_pnl, plot_pnl
import plotly.io as pio
pio.renderers.default = 'notebook'

---
## 步骤 1: 数据加载

复用了 DataLoader → Cleaner 的标准管道。
注意这里和 Notebook 01 完全相同的加载方式。

**关键洞察**: 这就是合约(Contract)的力量——
DataLoader 定义了统一的输出格式，
下游模块不需要关心数据从哪来。

In [ ]:
# 加载 OWID 中国数据
loader = create_loader('owid')
df = loader.load_data()
df = clean_data(df)

print(f"✓ 数据就绪: {len(df)} 行, {df['timestamp'].min().year}-{df['timestamp'].max().year}")
print(f"  平均负荷: {df['load_mw'].mean():.0f} MW")
df.head()

---
## 步骤 2: 持续法预测

### 持续法是什么？

$$\text{forecast}_t = \text{actual}_{t-24}$$

**用昨天的实际值作为今天的预测。**

### 为什么这种方法不蠢？
电力负荷有极强的**日周期 (Diurnal Cycle)**：
- 每天下午3点左右是用电高峰（夏天开空调、冬天取暖）
- 每天凌晨3-4点是用电低谷
- 昨天的下午3点和今天的下午3点，负荷非常接近

持续法利用了这种周期性——不需要任何训练数据，不需要特征工程，
但它常常能做到 MAPE 3-5%（对日级数据甚至更好）。

### 为什么它有效？（数学解释）
电力负荷的时间序列可以分解为：
$$L(t) = T(t) + S(t) + C(t) + R(t)$$
- $T(t)$: 趋势 (Trend) — 缓慢变化（经济增长）
- $S(t)$: 季节 (Seasonal) — 年/周模式
- $C(t)$: 周期 (Cyclical) — **日周期** ← 持续法捕捉这个
- $R(t)$: 残差 (Residual) — 无法预测的随机部分

持续法假设：$C(t) \approx C(t-24)$ 且其他分量变化慢。
对年级数据来说，$t$ 和 $t-24$ 都是年间隔……
但没关系，概念是一样的！

In [ ]:
# 执行持续法预测
forecast = persistence_forecast(df)

# 计算预测偏差
errors = (forecast - df['load_mw']).abs()
mae = errors.mean()

print(f"持续法预测 MAE: {mae:.0f} MW")
print(f"相对于平均负荷的比例: {mae / df['load_mw'].mean() * 100:.1f}%")
print()
print("📖 解读:")
print(f"  平均预测偏差 {mae:.0f} MW，大致相当于")
print(f"  一个中型城市的用电量。对于只用"昨天=今天"
print(f"  的简化模型来说，这个表现还不错。")

---
## 步骤 3: P&L 盈亏计算

### 模拟交易模型

假设你是一个**电力零售商**（像北京图迹服务的那些客户）：
1. 你需要在今天预测明天的用电量
2. 按照预测值在日前市场买入电力
3. 实际交付时，按真实负荷结算

**盈亏公式 (简化版):**
$$\text{P\&L}_t = -|\text{forecast}_t - \text{actual}_t| \times \frac{\text{price}}{1000}$$

偏差的绝对值 × 电价 = 每一时刻的损失。
负号表示：偏差越大，亏得越多。

> ⚠️ **注意**: 这是大幅简化的学习模型。
> 真实电力市场有：偏差容忍带、不平衡结算、辅助服务市场、
> 阻塞管理、节点电价等数十种复杂机制。
> 我们会在 Phase 2-3 逐步引入这些概念。

In [ ]:
# 计算累计盈亏
pnl = calculate_pnl(df['load_mw'], forecast, price_per_mwh=50)

print(f"累计模拟 P&L: {pnl.iloc[-1]:.2f}")
print()
print("📖 解读:")
print("  P&L 是负的——说明持续法预测有偏差，产生了"交易损失"。")
print("  但这完全正常！持续法只是基线，用来对比后续模型。")
print("  目标是: XGBoost 的 P&L > 持续法的 P&L")
print()
if pnl.iloc[-1] > pnl.iloc[0]:
    print("  🟢 累计 P&L 在上行——预测偏差在减小")
else:
    print("  🔴 累计 P&L 在下行——预测偏差在积压")

---
## 步骤 4: 端到端可视化

将数据→预测→盈亏整合到一张交互图中。

这是 **Phase 1 的最终输出**——证明整个管道是通的。

In [ ]:
# 绘制端到端结果图
fig = plot_pnl(df, forecast, pnl)
fig.show()

### 🎉 恭喜！你刚刚完成了第一个端到端管道！

尽管用的是最简单的模型，但你**看到了完整的链路**：

```
数据源 (OWID) → 加载 (DataLoader) → 清洗 (Cleaner)
  → 预测 (Persistence) → 交易 (P&L) → 可视化 (Plotly)
```

这就是**行走骨架**——骨架已就位，接下来添肌肉（XGBoost 替换持续法）。

---

## 📝 学习笔记

1. ✅ 端到端管道已跑通——数据→预测→盈亏→图表
2. ✅ 持续法作为基线模型，简单但有效
3. ✅ P&L 计算建立了"预测好坏→交易盈亏"的量化连接
4. ⏭️ **下一步**: Notebook 04 → XGBoost 替换持续法，提升预测精度

## 🤔 反思题

1. 如果数据是小时级的，持续法还能用吗？
2. P&L 为什么会是负值？在什么情况下 P&L 会是正的？
3. 你能想到一个比持续法更好、但仍然很简单的预测方法吗？

### 思考题

1. 持续法预测 MAPE 大约是多少？你能用更简单的方法打败它吗？
2. 累计 PnL 曲线一直往下走说明了什么？真实市场中什么样的策略能赚钱？
3. 这个端到端管道缺少了什么？从数据到预测到 PnL，还差什么才能做交易决策？

